# Sewer &harr; aquifer exchange vs MODFLOW coupling frequency

The companion to `step3_plot_compare_coastal_exchange`. That notebook covers the
MODFLOW &harr; D-Flow FM (coastal) interface; this one covers the
MODFLOW &harr; SWMM (sewer) interface, so both halves of the coupling are
compared across the same scenario family.

Three quantities, each as a function of coupling frequency:

1. **cumulative sewer exchange** &mdash; from `swmm_q.npz`, the per-connection
   MODFLOW well flux. Net and gross are shown separately: they behave
   differently, because infiltration at the few connected junctions is partly
   offset by exfiltration elsewhere.
2. **cumulative outfall volume** &mdash; from `swmm.out`. This is the quantity
   that drives the D-Flow FM `[SourceSink]` tracer, so any frequency dependence
   here propagates to the sewage plume.
3. **connected fraction** &mdash; from `pipe_state.npz`, how many junctions have
   the water table above the pipe invert. A purely numerical diagnostic: coarse
   coupling samples the tide less often, which can change how often junctions
   are seen as connected.

> `swmm_q` is sampled once per **coupling step**, so the number of samples
> differs by up to 96x across scenarios (89 at 01.00D, 8,544 at 15.00M).
> Everything below is integrated against each scenario's own coupling interval
> &mdash; comparing raw sums would be badly misleading.

In [ ]:
%matplotlib inline
import sys
import pathlib as pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import flopy.plot.styles as styles

In [ ]:
sys.path.append("../common")
from liss_settings import (
    get_scenario_name,
    get_results_path,
    fig_ext,
    transparent,
)

#### Scenario family

In [ ]:
# ---- held fixed across the comparison -------------------------------------
domain = "gp"
resolution = "high"              # this notebook compares ACROSS coupling frequency,
n_connections = 244                # so resolution and connection count are fixed
# ----------------------------------------------------------------------------

# coupling frequencies to compare, in hours
couple_freqs = [24.0, 8.0, 4.0, 2.0, 1.0, 0.5, 0.25]

# gp_sewer.inp geometry, for expressing the exchange in the units sewer
# standards use (gallons per day per inch-diameter per mile)
PIPE_MILES = 56638.0 / 5280.0      # 56,638 ft of pipe
PIPE_D_IN = 8.0                    # median diameter, 0.2032 m
GAL2FT3 = 0.133681

fig_ws = pl.Path("figures")
fig_ws.mkdir(exist_ok=True, parents=True)


def freq_label(freq):
    if freq == 24.0:
        return f"{freq / 24:>2.0f} Day "
    if freq >= 1.0:
        return f"{freq:>2.0f} Hour"
    return f"{freq * 60:>2.0f} Min."

#### Load every scenario that has been run

Scenarios that have not been run are reported and skipped, so the comparison
still plots with a partial family.

In [ ]:
scen = {}
for freq in couple_freqs:
    name = get_scenario_name(domain, resolution, freq, n_connections)
    ws = get_results_path(domain, resolution, freq, n_connections)
    if not (ws / "swmm_q.npz").is_file():
        print(f"  skipping {name} - not run yet ({ws})")
        continue

    q = np.load(ws / "swmm_q.npz")
    keys = sorted(q.files, key=int)
    # (nstep, nconn) MODFLOW well flux in ft3/d; stored as -Q, so a NEGATIVE sum
    # means water is moving INTO the sewer
    arr = np.stack([q[k] for k in keys])

    # integrate against THIS scenario's coupling interval, not the sample count
    dt_days = freq / 24.0
    t_days = np.arange(1, len(keys) + 1) * dt_days

    entry = {
        "freq": freq,
        "label": freq_label(freq),
        "ws": ws,
        "nstep": len(keys),
        "dt_days": dt_days,
        "t_days": t_days,
        # net: signed, so infiltration and exfiltration cancel
        "cum_net_ft3": np.cumsum(-arr.sum(axis=1) * dt_days),
        # gross: total water crossing the pipe wall either way
        "cum_gross_ft3": np.cumsum(np.abs(arr).sum(axis=1) * dt_days),
    }

    ps = ws / "pipe_state.npz"
    if ps.is_file():
        s = np.load(ps)
        st = np.stack([s[k] for k in sorted(s.files, key=int)])
        # columns: [inactive MODFLOW cells, disconnected, dry pipes]
        entry["connected"] = (n_connections - st[:, 1]) / n_connections * 100.0
    else:
        entry["connected"] = None      # run predates the pipe_state output

    scen[name] = entry
    print(f"  {name}: {entry['nstep']:,} steps, dt={dt_days*24:.2f} h, "
          f"net {entry['cum_net_ft3'][-1]:,.0f} ft3")

assert scen, "no scenarios have been run - run step2_run_coupled_models first"
names = list(scen)
colors = list(mcolors.TABLEAU_COLORS.values())
line_styles = ["-", "--", "-.", ":", (0, (3, 10, 1, 10)), (0, (3, 1, 1, 1)), (0, (5, 1))]

#### Outfall discharge

Read from each scenario's saved `swmm.out`. Unlike the exchange this is on
SWMM's own reporting step, which is the same for every scenario, so the series
are directly comparable.

In [ ]:
import pyswmm
from swmm.toolkit.shared_enum import NodeAttribute

outfall_node = "O3"
for name, e in scen.items():
    out = e["ws"] / "swmm.out"
    if not out.is_file():
        e["outfall"] = None
        continue
    with pyswmm.Output(str(out)) as o:
        s = o.node_series(outfall_node, NodeAttribute.TOTAL_INFLOW)
    # pyswmm hands back datetime objects, so an untyped np.array would be dtype=object
    t = np.array(list(s.keys()), dtype="datetime64[s]")
    v = np.array(list(s.values()))                    # m3/s
    dt_s = np.diff(t).astype("timedelta64[s]").astype(float)
    dt_s = np.concatenate([[dt_s[0]], dt_s])
    e["outfall_t"] = (t - t[0]) / np.timedelta64(1, "D")
    e["outfall_q"] = v
    e["outfall_cum_m3"] = np.cumsum(v * dt_s)
    print(f"  {name}: outfall {len(v):,} points, "
          f"cumulative {e['outfall_cum_m3'][-1]:,.0f} m3")

#### Cumulative sewer exchange

In [ ]:
with styles.USGSMap():
    fig, axs = plt.subplots(2, 1, figsize=(7.5, 7.0), layout="constrained", sharex=True)

    for i, name in enumerate(names):
        e = scen[name]
        axs[0].plot(e["t_days"], e["cum_net_ft3"], lw=1.0,
                    ls=line_styles[i % len(line_styles)],
                    color=colors[i % len(colors)], label=e["label"])
        axs[1].plot(e["t_days"], e["cum_gross_ft3"], lw=1.0,
                    ls=line_styles[i % len(line_styles)],
                    color=colors[i % len(colors)], label=e["label"])

    axs[0].set_ylabel("Cumulative NET exchange, ft$^3$\n(positive = into the sewer)")
    axs[1].set_ylabel("Cumulative GROSS exchange, ft$^3$")
    axs[1].set_xlabel("Time, days")
    for ax, head in zip(axs, ("Net (infiltration minus exfiltration)", "Gross (both directions)")):
        ax.axhline(0.0, lw=0.5, ls="--", color="black")
        styles.heading(ax, heading=head)
    styles.graph_legend(ax=axs[0], loc="upper left", title="none", ncol=2)
    fig.savefig(fig_ws / f"swmm_exchange_cumulative_{resolution}{fig_ext}",
                dpi=300, transparent=transparent)
print("wrote", fig_ws / f"swmm_exchange_cumulative_{resolution}{fig_ext}")

#### Outfall volume and connected fraction

In [ ]:
have_outfall = [n for n in names if scen[n].get("outfall_cum_m3") is not None]
have_conn = [n for n in names if scen[n].get("connected") is not None]

with styles.USGSMap():
    fig, axs = plt.subplots(2, 1, figsize=(7.5, 7.0), layout="constrained")

    for i, name in enumerate(have_outfall):
        e = scen[name]
        axs[0].plot(e["outfall_t"], e["outfall_cum_m3"], lw=1.0,
                    ls=line_styles[i % len(line_styles)],
                    color=colors[i % len(colors)], label=e["label"])
    axs[0].set_ylabel("Cumulative outfall volume, m$^3$")
    axs[0].set_xlabel("Time, days")
    styles.heading(axs[0], heading=f"Outfall {outfall_node} (drives the D-Flow FM tracer)")
    if have_outfall:
        styles.graph_legend(ax=axs[0], loc="upper left", title="none", ncol=2)

    for i, name in enumerate(have_conn):
        e = scen[name]
        axs[1].plot(e["t_days"], e["connected"], lw=0.8,
                    ls=line_styles[i % len(line_styles)],
                    color=colors[i % len(colors)], label=e["label"])
    axs[1].set_ylabel(f"Connected junctions, % of {n_connections}")
    axs[1].set_xlabel("Time, days")
    styles.heading(axs[1], heading="Junctions with the water table above the pipe invert")
    if not have_conn:
        axs[1].text(0.5, 0.5, "no pipe_state.npz - runs predate that output",
                    ha="center", va="center", transform=axs[1].transAxes)

    fig.savefig(fig_ws / f"swmm_outfall_connected_{resolution}{fig_ext}",
                dpi=300, transparent=transparent)
print("wrote", fig_ws / f"swmm_outfall_connected_{resolution}{fig_ext}")

#### Summary

Totals at the end of the run, with the exchange also expressed in the units
sewer standards use. Allowable infiltration is 50-200 gpd/in-diameter/mile for
acceptance testing of new gravity sanitary sewers, and 200-1000 for the I&I
design allowance in existing systems.

In [ ]:
rows = []
for name in names:
    e = scen[name]
    days = e["t_days"][-1]
    net_ft3 = e["cum_net_ft3"][-1]
    rate = (net_ft3 / days) / GAL2FT3 / (PIPE_D_IN * PIPE_MILES)
    rows.append({
        "scenario": name,
        "coupling": e["label"].strip(),
        "steps": e["nstep"],
        "net ft3": net_ft3,
        "gross ft3": e["cum_gross_ft3"][-1],
        "gpd/in/mi": rate,
        "connected %": np.mean(e["connected"]) if e["connected"] is not None else np.nan,
        "outfall m3": e["outfall_cum_m3"][-1] if e.get("outfall_cum_m3") is not None else np.nan,
    })
df = pd.DataFrame(rows).set_index("scenario")
ref = df.iloc[-1]        # finest coupling as the reference
df["net vs finest %"] = 100.0 * (df["net ft3"] - ref["net ft3"]) / abs(ref["net ft3"])
df["outfall vs finest %"] = 100.0 * (df["outfall m3"] - ref["outfall m3"]) / ref["outfall m3"]
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
df